# Grafik Bab 4 dari Dashboard Terakhir

Run acuan PUBLISHED: `outputs/reports/active_gold_run.json` (session `excel-20260826T130441+0000-9f3a9b`, run `07fc979c-5e04-4ca2-b599-b72ad80445c7`).
Jalankan dari root `C:\Users\hm\keuangan-final`. Output PNG dipakai di skripsi `gambar/bab4/`.


In [ ]:
import json
rep = json.load(open("outputs/reports/active_gold_run.json", encoding="utf-8"))
assert rep["status"] == "PUBLISHED"
print(rep["dataset_session_id"], rep["pipeline_run_id"])


In [ ]:
import pandas as pd
BASE = "data/generated/gold/excel-20260826T130441+0000-9f3a9b/07fc979c-5e04-4ca2-b599-b72ad80445c7"
iku = pd.read_csv(f"{BASE}/vw_dashboard_iku_anggaran.csv")
per_tahun = iku.groupby("tahun", as_index=False)["jumlah_weighted_iku"].sum().sort_values("tahun")
print(per_tahun.to_string(index=False))
print("total:", float(per_tahun.jumlah_weighted_iku.sum()))  # 517361656.0
assert abs(float(per_tahun.jumlah_weighted_iku.sum()) - 517361656) < 5


In [ ]:
import matplotlib.pyplot as plt
OUT = "D:/Hermawan only/taaaaaaaa-sd-itera/gambar/bab4"
plt.rcParams.update({"font.family": "serif", "font.size": 9})
def bar(x, y, title, xl, yl, fn, rot=30):
    import pandas as pd
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.bar(list(x), list(y)); ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel(yl)
    plt.setp(ax.get_xticklabels(), rotation=rot, ha="right"); fig.tight_layout()
    fig.savefig(f"{OUT}/{fn}", dpi=300); plt.close(fig)
    pd.DataFrame({"label": list(x), "nilai": list(y)}).to_csv(f"{OUT}/data/{fn.replace('.png','.csv')}", index=False)
    print("saved", fn)
bar(per_tahun["tahun"].astype(str), per_tahun["jumlah_weighted_iku"], "Total Anggaran Per Tahun", "Tahun", "Rupiah", "grafik_total_per_tahun.png", 0)


In [ ]:
per_iku = iku.groupby("kode_iku", as_index=False)["jumlah_weighted_iku"].sum().sort_values("jumlah_weighted_iku", ascending=False)
bar(per_iku["kode_iku"], per_iku["jumlah_weighted_iku"], "Anggaran Per Indikator Kinerja", "Kode IKU", "Rupiah", "grafik_per_iku.png", 60)
mq = pd.read_csv(f"{BASE}/vw_dashboard_mapping_quality.csv")
mq["label"] = mq["mapping_status"] + " | " + mq["data_quality_flag"]
bar(mq["label"], mq["item_count"], "Kualitas Pemetaan Item", "Status", "Item", "grafik_kualitas_mapping.png", 20)
rekon = pd.read_csv(f"{BASE}/vw_dashboard_reconciliation.csv")
assert int((rekon["selisih"] != 0).sum()) == 0
bar(rekon["source_file"].str.slice(0, 28), rekon["total_detail"], "Rekonsiliasi Per Berkas", "Berkas", "Rupiah", "grafik_rekonsiliasi_berkas.png", 25)


In [ ]:
fact = pd.read_csv(f"{BASE}/fact_anggaran.csv")
assert len(fact) == 607
dim_akun = pd.read_csv(f"{BASE}/dim_akun.csv")
akun = fact.groupby("akun_key", as_index=False)["jumlah_weighted_iku"].sum().merge(dim_akun[["akun_key","nama_akun"]], on="akun_key", how="left").sort_values("jumlah_weighted_iku", ascending=False)
bar(akun["nama_akun"].fillna("tanpa akun").str.slice(0,30), akun["jumlah_weighted_iku"], "Komposisi Akun Belanja", "Akun", "Rupiah", "grafik_komposisi_akun.png")
dim_keg = pd.read_csv(f"{BASE}/dim_kegiatan.csv")
keg = fact.groupby("kegiatan_key", as_index=False)["jumlah_weighted_iku"].sum().merge(dim_keg[["kegiatan_key","kegiatan"]], on="kegiatan_key", how="left").sort_values("jumlah_weighted_iku", ascending=False).head(15)
bar(keg["kegiatan"].fillna("tanpa kegiatan").str.slice(0,28), keg["jumlah_weighted_iku"], "Anggaran Kegiatan Terbesar", "Kegiatan", "Rupiah", "grafik_per_kegiatan.png", 25)


In [ ]:
inv = pd.read_csv("outputs/bab4_evidence_final/audit_20260731T152553Z/01_perancangan/411_analisis_sumber/workbook_inventory.csv")
bar(inv["source_file"].str.slice(0,30), inv["filled_cells"], "Sel Terisi Per Berkas", "Berkas", "Sel", "grafik_sel_per_berkas.png", 25)
bar(inv["source_file"].str.slice(0,30), inv["formula_count"], "Formula Per Berkas", "Berkas", "Formula", "grafik_formula_per_berkas.png", 25)
err = pd.read_csv("outputs/bab4_evidence_final/audit_20260731T152553Z/01_perancangan/411_analisis_sumber/formula_error_detail.csv")
bar(err["source_sheet"], err["formula_error_count"], "Galat Formula Per Lembar", "Lembar", "Sel", "grafik_error_per_lembar.png", 0)
rt = pd.read_csv("outputs/bab4_evidence_final/audit_20260731T152553Z/02_implementasi/422_bronze_dan_silver/silver_row_type_summary.csv")
bar(rt["row_type"], rt["row_count"], "Sebaran Jenis Baris", "Jenis", "Baris", "grafik_jenis_baris.png", 0)
sr = pd.read_csv("outputs/bab4_evidence_final/audit_20260731T152553Z/02_implementasi/422_bronze_dan_silver/silver_source_role_summary.csv")
bar(sr["source_role"], sr["row_count"], "Sebaran Peran Sumber", "Peran", "Baris", "grafik_peran_sumber.png", 0)
